# 🤖 FarmTech Solutions - Modelos Preditivos de Rendimento

## Parte 3: Previsão de Rendimento de Safras com Machine Learning

**Objetivo:** Desenvolver e comparar 5 modelos de regressão para prever o rendimento (Yield) das safras.

**Modelos implementados:**
1. Regressão Linear
2. Random Forest Regressor
3. Gradient Boosting Regressor  
4. Support Vector Regression (SVR)
5. Multi-Layer Perceptron (MLP Regressor)

**Métricas de avaliação:**
- R² (Coefficient of Determination)
- RMSE (Root Mean Squared Error)
- MAE (Mean Absolute Error)
- MAPE (Mean Absolute Percentage Error)

---

## 📦 1. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np
import pickle

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modelos de Regressão
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# Métricas
from sklearn.metrics import (
    mean_squared_error, 
    mean_absolute_error, 
    r2_score,
    mean_absolute_percentage_error
)

# Configurações
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 7)
sns.set_style('whitegrid')
sns.set_palette('husl')

print('✅ Bibliotecas importadas com sucesso!')

## 📊 2. Carregamento e Preparação dos Dados

In [ ]:
# Carregar dataset
df = pd.read_csv('../data/crop_yield.csv')

print(f'📁 Dataset carregado: {df.shape[0]} linhas, {df.shape[1]} colunas')
print(f'\nPrimeiras linhas:')
df.head()

In [ ]:
# Verificar informações do dataset
print('ℹ️ Informações do Dataset:')
print('='*70)
df.info()

print('\n📊 Estatísticas:')
df.describe().T

### 2.1 Preparação das Features

In [ ]:
# Separar features e target
# Features numéricas
numeric_features = ['Precipitation (mm day-1)', 'Specific Humidity at 2 Meters (g/kg)', 
                   'Relative Humidity at 2 Meters (%)', 'Temperature at 2 Meters (C)']

X_numeric = df[numeric_features].copy()

# Encoding da variável categórica 'Crop'
le = LabelEncoder()
df['Crop_Encoded'] = le.fit_transform(df['Crop'])

# Combinar features
X = df[numeric_features + ['Crop_Encoded']].copy()
y = df['Yield'].copy()

print('🔢 Features preparadas:')
print(f'  - Features numéricas: {numeric_features}')
print(f'  - Feature categórica: Crop (encoded)')
print(f'\n📊 Shape das features: {X.shape}')
print(f'📊 Shape do target: {y.shape}')

print(f'\n🌾 Encoding das culturas:')
for i, crop in enumerate(le.classes_):
    print(f'  {crop}: {i}')

### 2.2 Divisão Treino/Teste

In [ ]:
# Split treino e teste (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('✅ Dados divididos em treino e teste')
print(f'\n📊 Tamanhos:')
print(f'  - Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  - Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.1f}%)')

print(f'\n📈 Estatísticas do target:')
print(f'  - Yield médio (treino): {y_train.mean():.2f}')
print(f'  - Yield médio (teste):  {y_test.mean():.2f}')

### 2.3 Normalização

In [ ]:
# Normalizar features (importante para SVR e MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Features normalizadas com StandardScaler')
print('\n💡 Nota: Normalização é especialmente importante para SVR e MLP')

## 🤖 3. Desenvolvimento dos Modelos

### 3.1 Modelo 1: Regressão Linear

In [ ]:
print('🔄 Treinando Modelo 1: Regressão Linear...')

# Treinar modelo
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predições
y_pred_lr_train = lr_model.predict(X_train)
y_pred_lr_test = lr_model.predict(X_test)

# Métricas
lr_r2_train = r2_score(y_train, y_pred_lr_train)
lr_r2_test = r2_score(y_test, y_pred_lr_test)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr_test))
lr_mae = mean_absolute_error(y_test, y_pred_lr_test)
lr_mape = mean_absolute_percentage_error(y_test, y_pred_lr_test) * 100

print('✅ Regressão Linear treinada!')
print(f'\n📊 Resultados:')
print(f'  - R² (treino): {lr_r2_train:.4f}')
print(f'  - R² (teste):  {lr_r2_test:.4f}')
print(f'  - RMSE: {lr_rmse:.2f}')
print(f'  - MAE:  {lr_mae:.2f}')
print(f'  - MAPE: {lr_mape:.2f}%')

# Coeficientes
print(f'\n🔍 Coeficientes das features:')
feature_names = numeric_features + ['Crop_Encoded']
for feat, coef in zip(feature_names, lr_model.coef_):
    print(f'  {feat}: {coef:.4f}')
print(f'  Intercept: {lr_model.intercept_:.4f}')

### 3.2 Modelo 2: Random Forest Regressor

In [ ]:
print('🔄 Treinando Modelo 2: Random Forest Regressor...')

# Treinar modelo
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predições
y_pred_rf_train = rf_model.predict(X_train)
y_pred_rf_test = rf_model.predict(X_test)

# Métricas
rf_r2_train = r2_score(y_train, y_pred_rf_train)
rf_r2_test = r2_score(y_test, y_pred_rf_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf_test))
rf_mae = mean_absolute_error(y_test, y_pred_rf_test)
rf_mape = mean_absolute_percentage_error(y_test, y_pred_rf_test) * 100

print('✅ Random Forest treinado!')
print(f'\n📊 Resultados:')
print(f'  - R² (treino): {rf_r2_train:.4f}')
print(f'  - R² (teste):  {rf_r2_test:.4f}')
print(f'  - RMSE: {rf_rmse:.2f}')
print(f'  - MAE:  {rf_mae:.2f}')
print(f'  - MAPE: {rf_mape:.2f}%')

# Importância das features
print(f'\n🔍 Importância das features:')
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance.to_string(index=False))

### 3.3 Modelo 3: Gradient Boosting Regressor

In [ ]:
print('🔄 Treinando Modelo 3: Gradient Boosting Regressor...')

# Treinar modelo
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, 
                                     max_depth=5, random_state=42)
gb_model.fit(X_train, y_train)

# Predições
y_pred_gb_train = gb_model.predict(X_train)
y_pred_gb_test = gb_model.predict(X_test)

# Métricas
gb_r2_train = r2_score(y_train, y_pred_gb_train)
gb_r2_test = r2_score(y_test, y_pred_gb_test)
gb_rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb_test))
gb_mae = mean_absolute_error(y_test, y_pred_gb_test)
gb_mape = mean_absolute_percentage_error(y_test, y_pred_gb_test) * 100

print('✅ Gradient Boosting treinado!')
print(f'\n📊 Resultados:')
print(f'  - R² (treino): {gb_r2_train:.4f}')
print(f'  - R² (teste):  {gb_r2_test:.4f}')
print(f'  - RMSE: {gb_rmse:.2f}')
print(f'  - MAE:  {gb_mae:.2f}')
print(f'  - MAPE: {gb_mape:.2f}%')

# Importância das features
print(f'\n🔍 Importância das features:')
feature_importance_gb = pd.DataFrame({
    'Feature': feature_names,
    'Importance': gb_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(feature_importance_gb.to_string(index=False))

### 3.4 Modelo 4: Support Vector Regression (SVR)

In [ ]:
print('🔄 Treinando Modelo 4: Support Vector Regression (SVR)...')

# Treinar modelo (usando dados normalizados)
svr_model = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
svr_model.fit(X_train_scaled, y_train)

# Predições
y_pred_svr_train = svr_model.predict(X_train_scaled)
y_pred_svr_test = svr_model.predict(X_test_scaled)

# Métricas
svr_r2_train = r2_score(y_train, y_pred_svr_train)
svr_r2_test = r2_score(y_test, y_pred_svr_test)
svr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_svr_test))
svr_mae = mean_absolute_error(y_test, y_pred_svr_test)
svr_mape = mean_absolute_percentage_error(y_test, y_pred_svr_test) * 100

print('✅ SVR treinado!')
print(f'\n📊 Resultados:')
print(f'  - R² (treino): {svr_r2_train:.4f}')
print(f'  - R² (teste):  {svr_r2_test:.4f}')
print(f'  - RMSE: {svr_rmse:.2f}')
print(f'  - MAE:  {svr_mae:.2f}')
print(f'  - MAPE: {svr_mape:.2f}%')

print(f'\n⚙️ Hiperparâmetros:')
print(f'  - Kernel: {svr_model.kernel}')
print(f'  - C: {svr_model.C}')
print(f'  - Gamma: {svr_model.gamma}')
print(f'  - Epsilon: {svr_model.epsilon}')

### 3.5 Modelo 5: Multi-Layer Perceptron (MLP Regressor)

In [ ]:
print('🔄 Treinando Modelo 5: Multi-Layer Perceptron (MLP)...')

# Treinar modelo (usando dados normalizados)
mlp_model = MLPRegressor(hidden_layer_sizes=(100, 50), activation='relu', 
                        solver='adam', max_iter=1000, random_state=42)
mlp_model.fit(X_train_scaled, y_train)

# Predições
y_pred_mlp_train = mlp_model.predict(X_train_scaled)
y_pred_mlp_test = mlp_model.predict(X_test_scaled)

# Métricas
mlp_r2_train = r2_score(y_train, y_pred_mlp_train)
mlp_r2_test = r2_score(y_test, y_pred_mlp_test)
mlp_rmse = np.sqrt(mean_squared_error(y_test, y_pred_mlp_test))
mlp_mae = mean_absolute_error(y_test, y_pred_mlp_test)
mlp_mape = mean_absolute_percentage_error(y_test, y_pred_mlp_test) * 100

print('✅ MLP treinado!')
print(f'\n📊 Resultados:')
print(f'  - R² (treino): {mlp_r2_train:.4f}')
print(f'  - R² (teste):  {mlp_r2_test:.4f}')
print(f'  - RMSE: {mlp_rmse:.2f}')
print(f'  - MAE:  {mlp_mae:.2f}')
print(f'  - MAPE: {mlp_mape:.2f}%')

print(f'\n⚙️ Configuração da rede:')
print(f'  - Camadas ocultas: {mlp_model.hidden_layer_sizes}')
print(f'  - Ativação: {mlp_model.activation}')
print(f'  - Solver: {mlp_model.solver}')
print(f'  - Iterações realizadas: {mlp_model.n_iter_}')

## 📊 4. Comparação dos Modelos

In [ ]:
# Criar tabela comparativa
results = pd.DataFrame({
    'Modelo': ['Linear Regression', 'Random Forest', 'Gradient Boosting', 'SVR', 'MLP'],
    'R² (Treino)': [lr_r2_train, rf_r2_train, gb_r2_train, svr_r2_train, mlp_r2_train],
    'R² (Teste)': [lr_r2_test, rf_r2_test, gb_r2_test, svr_r2_test, mlp_r2_test],
    'RMSE': [lr_rmse, rf_rmse, gb_rmse, svr_rmse, mlp_rmse],
    'MAE': [lr_mae, rf_mae, gb_mae, svr_mae, mlp_mae],
    'MAPE (%)': [lr_mape, rf_mape, gb_mape, svr_mape, mlp_mape]
})

# Calcular overfitting (diferença entre treino e teste)
results['Overfitting'] = (results['R² (Treino)'] - results['R² (Teste)']).abs()

print('📊 COMPARAÇÃO DOS MODELOS')
print('='*110)
print(results.to_string(index=False))
print('='*110)

# Identificar o melhor modelo
best_model_idx = results['R² (Teste)'].idxmax()
best_model_name = results.loc[best_model_idx, 'Modelo']
best_r2 = results.loc[best_model_idx, 'R² (Teste)']

print(f'\n🏆 MELHOR MODELO: {best_model_name}')
print(f'   R² (Teste): {best_r2:.4f}')

In [ ]:
# Visualização comparativa
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Comparação R²
x_pos = np.arange(len(results))
width = 0.35

axes[0, 0].bar(x_pos - width/2, results['R² (Treino)'], width, 
              label='Treino', alpha=0.8, color='skyblue', edgecolor='black')
axes[0, 0].bar(x_pos + width/2, results['R² (Teste)'], width, 
              label='Teste', alpha=0.8, color='coral', edgecolor='black')
axes[0, 0].set_xlabel('Modelo', fontsize=12)
axes[0, 0].set_ylabel('R²', fontsize=12)
axes[0, 0].set_title('Comparação R² - Treino vs Teste', fontweight='bold', fontsize=14)
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(results['Modelo'], rotation=45, ha='right')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3, axis='y')

# 2. Comparação RMSE
axes[0, 1].bar(results['Modelo'], results['RMSE'], 
              color='lightgreen', alpha=0.8, edgecolor='black')
axes[0, 1].set_xlabel('Modelo', fontsize=12)
axes[0, 1].set_ylabel('RMSE', fontsize=12)
axes[0, 1].set_title('Comparação RMSE (quanto menor, melhor)', fontweight='bold', fontsize=14)
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. Comparação MAE
axes[1, 0].bar(results['Modelo'], results['MAE'], 
              color='plum', alpha=0.8, edgecolor='black')
axes[1, 0].set_xlabel('Modelo', fontsize=12)
axes[1, 0].set_ylabel('MAE', fontsize=12)
axes[1, 0].set_title('Comparação MAE (quanto menor, melhor)', fontweight='bold', fontsize=14)
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(alpha=0.3, axis='y')

# 4. Comparação MAPE
axes[1, 1].bar(results['Modelo'], results['MAPE (%)'], 
              color='gold', alpha=0.8, edgecolor='black')
axes[1, 1].set_xlabel('Modelo', fontsize=12)
axes[1, 1].set_ylabel('MAPE (%)', fontsize=12)
axes[1, 1].set_title('Comparação MAPE (quanto menor, melhor)', fontweight='bold', fontsize=14)
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 📈 5. Análise de Predições vs Valores Reais

In [ ]:
# Gráficos de predição vs real para todos os modelos
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

predictions = [
    ('Linear Regression', y_pred_lr_test),
    ('Random Forest', y_pred_rf_test),
    ('Gradient Boosting', y_pred_gb_test),
    ('SVR', y_pred_svr_test),
    ('MLP', y_pred_mlp_test)
]

for idx, (name, y_pred) in enumerate(predictions):
    # Scatter plot
    axes[idx].scatter(y_test, y_pred, alpha=0.5, s=30, edgecolors='black', linewidth=0.5)
    
    # Linha de referência (predição perfeita)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    axes[idx].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Predição Perfeita')
    
    axes[idx].set_xlabel('Valor Real (Yield)', fontsize=11)
    axes[idx].set_ylabel('Valor Predito (Yield)', fontsize=11)
    axes[idx].set_title(f'{name}\nR² = {results.loc[idx, "R² (Teste)"]:.4f}', 
                       fontweight='bold', fontsize=12)
    axes[idx].legend(loc='upper left', fontsize=9)
    axes[idx].grid(alpha=0.3)

# Remover subplot extra
fig.delaxes(axes[5])

plt.suptitle('Predições vs Valores Reais - Conjunto de Teste', 
             fontweight='bold', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

## 🎯 6. Análise de Resíduos

In [ ]:
# Análise de resíduos para o melhor modelo
if best_model_name == 'Linear Regression':
    best_predictions = y_pred_lr_test
elif best_model_name == 'Random Forest':
    best_predictions = y_pred_rf_test
elif best_model_name == 'Gradient Boosting':
    best_predictions = y_pred_gb_test
elif best_model_name == 'SVR':
    best_predictions = y_pred_svr_test
else:  # MLP
    best_predictions = y_pred_mlp_test

residuals = y_test - best_predictions

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Distribuição dos resíduos
axes[0].hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Resíduo = 0')
axes[0].set_xlabel('Resíduos (Real - Predito)', fontsize=12)
axes[0].set_ylabel('Frequência', fontsize=12)
axes[0].set_title(f'Distribuição dos Resíduos - {best_model_name}', fontweight='bold', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# 2. Resíduos vs Predições
axes[1].scatter(best_predictions, residuals, alpha=0.5, s=30, edgecolors='black', linewidth=0.5)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2, label='Resíduo = 0')
axes[1].set_xlabel('Valores Preditos (Yield)', fontsize=12)
axes[1].set_ylabel('Resíduos (Real - Predito)', fontsize=12)
axes[1].set_title(f'Resíduos vs Predições - {best_model_name}', fontweight='bold', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'📊 Estatísticas dos Resíduos ({best_model_name}):')
print(f'  - Média: {residuals.mean():.4f} (idealmente próximo de 0)')
print(f'  - Desvio Padrão: {residuals.std():.4f}')
print(f'  - Mínimo: {residuals.min():.4f}')
print(f'  - Máximo: {residuals.max():.4f}')

## 🔍 7. Importância das Features (Random Forest e Gradient Boosting)

In [ ]:
# Visualizar importância das features
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Random Forest
feature_importance.sort_values('Importance', ascending=True, inplace=False).plot(
    x='Feature', y='Importance', kind='barh', ax=ax1, 
    color='steelblue', alpha=0.8, edgecolor='black'
)
ax1.set_xlabel('Importância', fontsize=12)
ax1.set_ylabel('Feature', fontsize=12)
ax1.set_title('Importância das Features - Random Forest', fontweight='bold', fontsize=14)
ax1.legend().set_visible(False)
ax1.grid(alpha=0.3, axis='x')

# Gradient Boosting
feature_importance_gb.sort_values('Importance', ascending=True, inplace=False).plot(
    x='Feature', y='Importance', kind='barh', ax=ax2, 
    color='forestgreen', alpha=0.8, edgecolor='black'
)
ax2.set_xlabel('Importância', fontsize=12)
ax2.set_ylabel('Feature', fontsize=12)
ax2.set_title('Importância das Features - Gradient Boosting', fontweight='bold', fontsize=14)
ax2.legend().set_visible(False)
ax2.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 🔄 8. Validação Cruzada

In [ ]:
# Validação cruzada para todos os modelos
print('🔄 Realizando Validação Cruzada (5-Fold)...')
print('='*90)

models = [
    ('Linear Regression', LinearRegression(), X_train),
    ('Random Forest', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train),
    ('Gradient Boosting', GradientBoostingRegressor(n_estimators=100, random_state=42), X_train),
    ('SVR', SVR(kernel='rbf', C=100, gamma='scale'), X_train_scaled),
    ('MLP', MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42), X_train_scaled)
]

cv_results = []

for name, model, X_data in models:
    scores = cross_val_score(model, X_data, y_train, cv=5, scoring='r2', n_jobs=-1)
    cv_results.append({
        'Modelo': name,
        'CV R² Médio': scores.mean(),
        'CV R² Std': scores.std(),
        'Min': scores.min(),
        'Max': scores.max()
    })
    print(f'{name:20s} - R² médio: {scores.mean():.4f} (+/- {scores.std():.4f})')

cv_df = pd.DataFrame(cv_results)
print('\n✅ Validação cruzada concluída!')

In [ ]:
# Visualizar resultados da validação cruzada
fig, ax = plt.subplots(figsize=(14, 6))

x_pos = np.arange(len(cv_df))
ax.bar(x_pos, cv_df['CV R² Médio'], yerr=cv_df['CV R² Std'], 
       alpha=0.8, color='teal', edgecolor='black', capsize=10)
ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('R² (Validação Cruzada)', fontsize=12)
ax.set_title('Validação Cruzada (5-Fold) - R² Médio ± Desvio Padrão', 
             fontweight='bold', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(cv_df['Modelo'], rotation=45, ha='right')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 💾 9. Salvar o Melhor Modelo

In [ ]:
# Identificar e salvar o melhor modelo
if best_model_name == 'Linear Regression':
    best_model = lr_model
    best_scaler = None  # LR não usa normalização
elif best_model_name == 'Random Forest':
    best_model = rf_model
    best_scaler = None  # RF não precisa de normalização
elif best_model_name == 'Gradient Boosting':
    best_model = gb_model
    best_scaler = None  # GB não precisa de normalização
elif best_model_name == 'SVR':
    best_model = svr_model
    best_scaler = scaler  # SVR precisa de normalização
else:  # MLP
    best_model = mlp_model
    best_scaler = scaler  # MLP precisa de normalização

# Salvar modelo
with open('melhor_modelo_rendimento.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Salvar scaler se necessário
if best_scaler is not None:
    with open('scaler_features.pkl', 'wb') as f:
        pickle.dump(best_scaler, f)

# Salvar label encoder
with open('label_encoder_crop.pkl', 'wb') as f:
    pickle.dump(le, f)

print(f'✅ Melhor modelo salvo: {best_model_name}')
print(f'   Arquivo: melhor_modelo_rendimento.pkl')
if best_scaler is not None:
    print(f'   Scaler salvo: scaler_features.pkl')
print(f'   Label Encoder salvo: label_encoder_crop.pkl')

## 📝 10. Conclusões e Recomendações

In [ ]:
print('📊 CONCLUSÕES FINAIS')
print('='*90)

print(f'\n🏆 MELHOR MODELO: {best_model_name}')
print(f'\n📈 Desempenho do melhor modelo:')
best_results = results[results['Modelo'] == best_model_name].iloc[0]
print(f'  - R² (Teste):     {best_results["R² (Teste)"]:.4f}')
print(f'  - RMSE:           {best_results["RMSE"]:.2f}')
print(f'  - MAE:            {best_results["MAE"]:.2f}')
print(f'  - MAPE:           {best_results["MAPE (%)"]:.2f}%')
print(f'  - Overfitting:    {best_results["Overfitting"]:.4f} (diferença R² treino/teste)')

print(f'\n🔍 ANÁLISE COMPARATIVA:')
print(f'  1. Melhor R² (Teste):  {results.loc[results["R² (Teste)"].idxmax(), "Modelo"]} ({results["R² (Teste)"].max():.4f})')
print(f'  2. Menor RMSE:         {results.loc[results["RMSE"].idxmin(), "Modelo"]} ({results["RMSE"].min():.2f})')
print(f'  3. Menor MAE:          {results.loc[results["MAE"].idxmin(), "Modelo"]} ({results["MAE"].min():.2f})')
print(f'  4. Menor MAPE:         {results.loc[results["MAPE (%)"].idxmin(), "Modelo"]} ({results["MAPE (%)"].min():.2f}%)')
print(f'  5. Menor Overfitting:  {results.loc[results["Overfitting"].idxmin(), "Modelo"]} ({results["Overfitting"].min():.4f})')

print(f'\n💡 PRINCIPAIS INSIGHTS:')
if 'Random Forest' in best_model_name or 'Gradient Boosting' in best_model_name:
    top_feature = feature_importance.iloc[0]['Feature'] if best_model_name == 'Random Forest' else feature_importance_gb.iloc[0]['Feature']
    print(f'  - Feature mais importante: {top_feature}')
print(f'  - Modelos ensemble (RF e GB) geralmente apresentam melhor desempenho')
print(f'  - SVR e MLP requerem normalização dos dados')
print(f'  - Validação cruzada confirma a robustez dos resultados')

print(f'\n🎯 RECOMENDAÇÕES:')
print(f'  1. Usar o modelo {best_model_name} para predições em produção')
print(f'  2. Considerar feature engineering adicional para melhorar o desempenho')
print(f'  3. Realizar tuning de hiperparâmetros (GridSearch/RandomSearch)')
print(f'  4. Coletar mais dados para melhorar a generalização')
print(f'  5. Monitorar o desempenho do modelo em produção continuamente')

print('\n' + '='*90)
print('✅ ANÁLISE COMPLETA FINALIZADA!')
print('='*90)